# Anomaly Detection EDA

This notebook works in both Google Colab and local Jupyter.

- Colab uses the Google Drive parquet path
- Local mode uses `../dataset` and falls back to `./dataset`
- Iteration 1 focuses on schema validation and basic sanity checks


In [2]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def running_in_colab():
    return importlib.util.find_spec("google.colab") is not None


def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])
    return __import__(import_name)


IS_COLAB = running_in_colab()
duckdb = ensure_package("duckdb")
pd = ensure_package("pandas")
con = duckdb.connect()


def query_df(query):
    return con.sql(query).df()


print(f"Running in Colab: {IS_COLAB}")
print(f"Python executable: {sys.executable}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Pandas version: {pd.__version__}")


Running in Colab: False
Python executable: /Users/harrish/Desktop/practicum/.venv/bin/python
DuckDB version: 1.5.2
Pandas version: 3.0.3


In [3]:
if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    dataset_root = Path("/content/drive/MyDrive/Practicum/NGIDS/NGIDS-DS-v1/parquet")
else:
    dataset_root = Path("../dataset")
    if not dataset_root.exists():
        dataset_root = Path("dataset")

host_logs = dataset_root / "host_logs.parquet"
ground_truth = dataset_root / "ground_truth.parquet"

print(f"dataset_root: {dataset_root}")
print(f"host_logs parquet: {host_logs}")
print(f"ground_truth parquet: {ground_truth}")


dataset_root: ../dataset
host_logs parquet: ../dataset/host_logs.parquet
ground_truth parquet: ../dataset/ground_truth.parquet


## Iteration 1: Dataset Orientation

This section verifies row counts, schemas, sample rows, time ranges, and a few label-like columns.


In [4]:
query_df(f"""
SELECT 'host_logs' AS dataset, count(*) AS row_count FROM '{host_logs}'
UNION ALL
SELECT 'ground_truth' AS dataset, count(*) AS row_count FROM '{ground_truth}'
""")


,dataset,row_count
0,host_logs,90054239
1,ground_truth,313926


In [5]:
query_df(f"DESCRIBE SELECT * FROM '{host_logs}'")


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,TIME,YES,None,None,None
2,pro_id,BIGINT,YES,None,None,None
3,path,VARCHAR,YES,None,None,None
4,sys_call,BIGINT,YES,None,None,None
5,event_id,BIGINT,YES,None,None,None
6,attack_cat,VARCHAR,YES,None,None,None
7,attack_subcat,VARCHAR,YES,None,None,None
8,label,BIGINT,YES,None,None,None


In [6]:
query_df(f"SELECT * FROM '{host_logs}' LIMIT 10")


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label
0,2016-03-11,02:45:01,1830,/sbin/upstart-dbus-bridge,142,45354,normal,normal,0
1,2016-03-11,02:45:06,1804,/bin/dbus-daemon,256,45352,normal,normal,0
2,2016-03-11,02:45:06,2133,/usr/lib/i386-linux-gnu/gconf/gconfd-2,168,45372,normal,normal,0
3,2016-03-11,02:45:35,4528,/usr/bin/python3.4,3,39459,normal,normal,0
4,2016-03-11,02:45:44,1847,/usr/bin/ibus-daemon,102,37263,normal,normal,0
5,2016-03-11,02:45:44,1907,/usr/lib/ibus/ibus-ui-gtk3,168,37896,normal,normal,0
6,2016-03-11,02:45:44,1925,/usr/lib/ibus/ibus-engine-simple,168,37542,normal,normal,0
7,2016-03-11,02:45:44,4461,/usr/sbin/apache2,142,37647,normal,normal,0
8,2016-03-11,02:45:45,1081,/usr/bin/Xorg,102,37480,normal,normal,0
9,2016-03-11,02:45:11,3989,/sbin/auditd,256,45374,normal,normal,0


In [7]:
query_df(f"""
SELECT
    min(date) AS min_date,
    max(date) AS max_date,
    min(time) AS min_time,
    max(time) AS max_time,
    count(*) AS row_count,
    count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS distinct_rows,
    count(*) - count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS duplicate_rows
FROM '{host_logs}'
""")


,min_date,max_date,min_time,max_time,row_count,distinct_rows,duplicate_rows
0,2016-03-11,2016-03-16,00:00:00,23:59:59,90054239,89709941,344298


In [8]:
query_df(f"""
SELECT attack_cat, attack_subcat, label, count(*) AS n
FROM '{host_logs}'
GROUP BY 1, 2, 3
ORDER BY n DESC
LIMIT 20
""")


,attack_cat,attack_subcat,label,n
0,normal,normal,0,88791812
1,Exploits,Office Document Batch,1,276578
2,Exploits,Browser,1,152319
3,Exploits,Clientside,1,102893
4,Generic,IXIA Batch,1,79624
5,Exploits,Clientside Microsoft Office Batch,1,71920
6,Exploits,Clientside Microsoft Paint,1,71869
7,Backdoors,All Batch,1,70712
8,Exploits,Browser FTP Batch,1,48994
9,Shellcode,Linux Batch,1,44245


In [9]:
query_df(f"""
SELECT
    count(DISTINCT pro_id) AS distinct_pro_id,
    count(DISTINCT path) AS distinct_path,
    count(DISTINCT sys_call) AS distinct_sys_call,
    count(DISTINCT event_id) AS distinct_event_id,
    count(DISTINCT attack_cat) AS distinct_attack_cat,
    count(DISTINCT attack_subcat) AS distinct_attack_subcat,
    count(DISTINCT label) AS distinct_label
FROM '{host_logs}'
""")


,distinct_pro_id,distinct_path,distinct_sys_call,distinct_event_id,distinct_attack_cat,distinct_attack_subcat,distinct_label
0,5576,100,122,89709941,8,53,2


In [10]:
query_df(f"DESCRIBE SELECT * FROM '{ground_truth}'")


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,VARCHAR,YES,None,None,None
2,attack_cat,VARCHAR,YES,None,None,None
3,attack_subcat,VARCHAR,YES,None,None,None
4,attack_name,VARCHAR,YES,None,None,None
5,attack_refrence,VARCHAR,YES,None,None,None
6,ips,VARCHAR,YES,None,None,None


In [11]:
query_df(f"SELECT * FROM '{ground_truth}' LIMIT 10")


,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,9:21:36,Backdoors,All Batch,Cisco Network Registrar Default Credentials Ba...,CVE 2011-2024 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.0:0->10.40.85.32:0 175.45.176.0:...
1,2016-03-14,12:14:24,Backdoors,All Batch,BlackEnergy Botnet Command and Control Communi...,http://atlas-public.ec2.arbor.net/docs/BlackEn...,175.45.176.1:3495->10.40.85.32:58782
2,2016-03-15,4:19:12,Backdoors,All Batch,Backdoor: Cisco Prime LAN Management (https://...,CVE 2012-6392 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:13177->10.40.85.32:514
3,2016-03-14,9:36:00,Backdoors,All Batch,phpmyadmin 3.5.2.2 Backdoor Access and Code Ex...,CVE 2012-5159 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.2:0->10.40.85.32:0 175.45.176.2:...
4,2016-03-15,11:45:36,Backdoors,All Batch,Android AndroidKungFu Malware Command and Cont...,http://about-threats.trendmicro.com/malware.as...,175.45.176.3:61508->10.40.85.32:7500
5,2016-03-11,9:21:36,Backdoors,All Batch,Cisco Network Registrar Default Credentials Ba...,CVE 2011-2024 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.0:0->10.40.85.32:0 175.45.176.0:...
6,2016-03-14,12:57:36,Backdoors,All Batch,BlackEnergy Botnet Command and Control Communi...,http://atlas-public.ec2.arbor.net/docs/BlackEn...,175.45.176.2:30436->10.40.85.32:30566
7,2016-03-14,12:14:24,Backdoors,All Batch,Backdoor: Cisco Prime LAN Management (https://...,CVE 2012-6392 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:59379->10.40.85.32:514
8,2016-03-16,2:09:36,Backdoors,All Batch,phpmyadmin 3.5.2.2 Backdoor Access and Code Ex...,CVE 2012-5159 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.0:0->10.40.85.32:0 175.45.176.0:...
9,2016-03-15,6:28:48,Backdoors,All Batch,Android AndroidKungFu Malware Command and Cont...,http://about-threats.trendmicro.com/malware.as...,175.45.176.1:49989->10.40.85.32:7500


In [12]:
query_df(f"""
SELECT
    min(date) AS min_date,
    max(date) AS max_date,
    count(*) AS row_count,
    count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS distinct_rows,
    count(*) - count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS duplicate_rows,
    count(*) FILTER (WHERE time = 'Time') AS header_like_rows
FROM '{ground_truth}'
""")


,min_date,max_date,row_count,distinct_rows,duplicate_rows,header_like_rows
0,2016-03-11,2016-03-16,313926,311621,2305,26


In [13]:
query_df(f"""
SELECT attack_cat, count(*) AS n
FROM '{ground_truth}'
GROUP BY 1
ORDER BY n DESC
LIMIT 20
""")


,attack_cat,n
0,Exploits,158316
1,Exploits,73301
2,Malware,35903
3,Denial of Service,18702
4,Generic,11300
5,Denial of Service,6100
6,Shellcode,5302
7,Reconnaissance,1900
8,Worms,1301
9,Backdoors,1200


In [14]:
query_df(f"SELECT * FROM '{ground_truth}' WHERE time = 'Time' LIMIT 10")


,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,Time,Malware,Mobile Batch,Strike Name,Strike Reference,Strike Tuples
1,2016-03-11,Time,Denial of Service,Browser Batch,Strike Name,Strike Reference,Strike Tuples
2,2016-03-11,Time,Denial of Service,HTTP,Strike Name,Strike Reference,Strike Tuples
3,2016-03-11,Time,Malware,package,Strike Name,Strike Reference,Strike Tuples
4,2016-03-11,Time,Denial of Service,NetBIOS/SMB Batch,Strike Name,Strike Reference,Strike Tuples
5,2016-03-11,Time,Exploits,Browser,Strike Name,Strike Reference,Strike Tuples
6,2016-03-11,Time,Exploits,Browser,Strike Name,Strike Reference,Strike Tuples
7,2016-03-11,Time,Exploits,Browser,Strike Name,Strike Reference,Strike Tuples
8,2016-03-11,Time,Exploits,Browser,Strike Name,Strike Reference,Strike Tuples
9,2016-03-11,Time,Exploits,Clientside Microsoft Office Batch,Strike Name,Strike Reference,Strike Tuples
